In [2]:
import zipfile
import os
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.signal import convolve2d
import pathlib
from sklearn.preprocessing import LabelEncoder

C:\Users\thaim\anaconda3\envs\tf210\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
def get_efficientNetB3(
    input_size: int = 224,
    num_classes: int = 5,
    backbone_trainable: bool = False,
    dropout_rate: float = 0.5,
    l2_reg: float = 0.01,
    l1_activity_reg: float = 0.002,
    learning_rate: float = 1e-3,
):
    """
    Creates a transfer‐learning model for diabetic‐retinopathy classification
    based on EfficientNetB3 (pretrained on ImageNet). By default, the EfficientNet
    backbone is frozen (backbone_trainable=False). After initial training you can
    unfreeze a few top layers.

    Args:
      input_size:          Height & width of the input (they should be square).
      num_classes:         Number of output classes (e.g. 5 for DR grades 0–4).
      backbone_trainable:  If False, the ImageNet‐trained EfficientNetB3 layers
                           stay frozen. If True, all of them become trainable.
      dropout_rate:        Dropout rate applied just before the final dense.
      l2_reg:              L2 weight regularizer strength for the final Dense.
      l1_activity_reg:     L1 activity regularizer strength for the final Dense.
      learning_rate:       Learning rate for the Adamax optimizer.

    Returns:
      A compiled tf.keras.Model ready for fit(…).
    """

    # 1) Build the EfficientNetB3 base (exclude top, ImageNet weights)
    base = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=(input_size, input_size, 3),
        pooling="avg",
    )
    base.trainable = backbone_trainable

    # 2) Assemble the “head” on top of the frozen/finetunable backbone:
    inputs = tf.keras.Input(shape=(input_size, input_size, 3))

    # → Preprocessing (scales to ImageNet range)
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)

    # → Backbone features
    x = base(x)

    # → BatchNorm (helps stabilize)
    x = tf.keras.layers.BatchNormalization(momentum=0.99, epsilon=1e-3)(x)

    # → Dropout for regularization
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    # → Final Dense with both L2 on weights and L1 on activations
    x = tf.keras.layers.Dense(
        units=num_classes,
        activation="softmax",
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
        activity_regularizer=tf.keras.regularizers.l1(l1_activity_reg),
    )(x)

    model = tf.keras.Model(inputs, x, name="EffNetB3_transfer")

    # 3) Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adamax(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


In [4]:
model = get_efficientNetB3(backbone_trainable=True)

In [5]:
model.load_weights('models_weights/mdl_ENB3_v2_masked_weights.h5')

In [6]:
ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/preprocessed_2/prpc_ver2_sigmax10_gain2_masked_3540",
    batch_size=50,
    seed=42,
    label_mode = "categorical",
    image_size=(224,224)
)

Found 3540 files belonging to 5 classes.


In [7]:
model.evaluate(ds)

71/71 [==============================] - 25s 214ms/step - loss: 0.0435 - accuracy: 0.9949


[0.04352317750453949, 0.994915246963501]